# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [21]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import GridSearchCV

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier

In [8]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [9]:
spaceship.dropna(inplace = True)

**Perform Train Test Split**

In [10]:
features = spaceship.drop(columns = ["Transported"])
target = spaceship["Transported"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.2, random_state = 23)

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [12]:
def cabin_transform(df):
    df["Cabin"] = df["Cabin"].apply(lambda x: x.split("/")[0])
    return df

def remove_id(df):
    return df.drop(columns = ["PassengerId", "Name"])

X_train = cabin_transform(X_train)
X_train = remove_id(X_train)


encoder = OneHotEncoder(sparse_output = False)
encoder.fit(X_train[["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]])

X_train_encoded = encoder.transform(X_train[["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]])
X_train_encoded = pd.DataFrame(X_train_encoded, columns = encoder.get_feature_names_out(["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]))
X_train_final = pd.concat([X_train.reset_index(drop = True), X_train_encoded.reset_index(drop = True)], axis=1)
X_train_final = X_train_final.drop(columns = ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP'])

X_test = cabin_transform(X_test)
X_test = remove_id(X_test)
X_test_encoded = encoder.transform(X_test[["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]])
X_test_encoded = pd.DataFrame(X_test_encoded, columns = encoder.get_feature_names_out(["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]))
X_test_final = pd.concat([X_test.reset_index(drop = True), X_test_encoded.reset_index(drop = True)], axis=1)
X_test_final = X_test_final.drop(columns = ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP'])

In [13]:
X_train_final.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True
0,24.0,1.0,467.0,4.0,0.0,341.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
1,42.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,8.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
4,33.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


In [14]:
X_test_final.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True
0,33.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,26.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,26.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,32.0,1.0,1146.0,0.0,50.0,34.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


In [15]:
normalizer = MinMaxScaler()
normalizer.fit(X_train_final)

,feature_range,"(0, ...)"
,copy,True
,clip,False


In [16]:
X_train_norm = normalizer.transform(X_train_final)
X_train_norm = pd.DataFrame(X_train_norm, columns = X_train_final.columns)
X_train_norm.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True
0,0.303797,0.000116,0.015664,0.000326,0.0,0.016768,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
1,0.531646,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.025316,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,0.101266,0.000000,0.000000,0.000000,0.0,0.000000,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
4,0.417722,0.000000,0.000000,0.000000,0.0,0.000000,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


In [17]:
X_test_norm = normalizer.transform(X_test_final)
X_test_norm = pd.DataFrame(X_test_norm, columns = X_test_final.columns)
X_test_norm.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True
0,0.417722,0.000000,0.00000,0.0,0.000000,0.000000,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,0.329114,0.000000,0.00000,0.0,0.000000,0.000000,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.329114,0.000000,0.00000,0.0,0.000000,0.000000,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,0.405063,0.000116,0.03844,0.0,0.002231,0.001672,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [ ]:
bagging_class = BaggingClassifier(DecisionTreeClassifier(max_depth = 20),
                 n_estimators = 100, 
                 max_samples = 1000)

# this was the best model for me

bagging_class.fit(X_train_norm, y_train)

,estimator,DecisionTreeC...(max_depth=20)
,n_estimators,100
,max_samples,1000
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,None
,random_state,None
,verbose,0


- Evaluate your model

In [19]:

bagging_class.score(X_test_norm, y_test)

0.7881996974281392

**Grid/Random Search**

For this lab we will use Grid Search.

In [22]:
bag = BaggingClassifier(DecisionTreeClassifier())

- Define hyperparameters to fine tune.

In [24]:
grid = {
    "n_estimators" : [50, 100, 200, 500],
    "max_samples" : [500, 1000, 2000],
    "estimator__max_leaf_nodes" : [250, 500, 1000],
    "estimator__max_depth" : [10, 30, 50]
}

- Run Grid Search

In [ ]:
model = GridSearchCV(estimator = bag, param_grid = grid, cv = 3)

In [26]:
model.fit(X_train_norm, y_train)

,estimator,BaggingClassi...eClassifier())
,param_grid,"{'estimator__max_depth': [10, 30, ...], 'estimator__max_leaf_nodes': [250, 500, ...], 'max_samples': [500, 1000, ...], 'n_estimators': [50, 100, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [28]:
model.best_params_

{'estimator__max_depth': 10,
 'estimator__max_leaf_nodes': 250,
 'max_samples': 500,
 'n_estimators': 50}

In [27]:
best_model = model.best_estimator_

- Evaluate your model

In [29]:
best_model.score(X_test_norm, y_test)

0.791981845688351

We are improving it on a 0.003